# Site Assessment Data (SAD)
Site Assessment Data examines the presence of contaminating substances in the soil at a specific location. In this registration object, the investigation concerns the quality of terrestrial soils and drier bank areas, as well as groundwater. In addition, the nature of any contamination is determined: which substances occur in concentrations higher than the natural background value? The extent of the contamination is also assessed by comparing the measured concentrations with the permitted levels based on a national or local regulatory framework.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import brodata

In [ ]:
sad = brodata.sad.SiteAssessmentData.from_bro_id("SAD000000011742")

Plot the geometry of the Site Assessment Data, together with the measurementPoints.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
gpd.GeoDataFrame(geometry=[sad.geometry]).plot(ax=ax)
sad.measurementPoint.plot(ax=ax, color="red", markersize=50)
for index in sad.measurementPoint.index:
    point = sad.measurementPoint.geometry.loc[index]
    ax.annotate(index, (point.x, point.y), textcoords="offset points", xytext=(0,10), ha='center')
ax.axis("equal");

Show the contents of the attribute `measurementPoint`, which is a geopandas GeoDataFrame.

In [ ]:
sad.measurementPoint

Plot lithology logs for all measurement points.

In [ ]:
f, ax = plt.subplots(figsize=(15,6))
for i, name in enumerate(sad.measurementPoint.index):
    df = sad.measurementPoint.at[name,'DescriptiveBoreholeLog']['layer']
    brodata.plot.bro_lithology_advanced(df, x=i, width=0.6, soil_name_column='soilName', ax=ax, bro_id=name)
    # plot filter if available
    if isinstance(sad.measurementPoint.at[name,'filter'], pd.DataFrame):
        fs = sad.measurementPoint.at[name,'filter']
        for fi in fs.index:
            y = [ -fs.at[fi,'upperBoundary'], -fs.at[fi,'lowerBoundary'] ]
            ax.plot([i, i], y, color='k', linewidth=3, linestyle=':', solid_capstyle="butt")
ax.set_xlim(-0.5, len(sad.measurementPoint) - 0.5)
ax.set_xticks(range(len(sad.measurementPoint)))
ax.set_xticklabels(sad.measurementPoint.index, fontdict={'rotation':45, 'ha':'right'})
ax.set_ylim(-sad.measurementPoint['finalDepth'].max()-0.1, 0.0)
ax.set_axisbelow(True)
ax.grid(True)
ax.set_ylabel('Depth (m)');

The sampling analysis results for a specific measurement filter of one of the measurement points are stored as a pandas DataFrame.

In [ ]:
name = "1786310"
filters = sad.measurementPoint.at[name, "filter"]
df = filters.iloc[0]['groundwaterSampleAnalysis']

# add the parameter descriptionb
parameter_list = brodata.gar.get_parameter_list()
# add a description when parameter is in parameter list
df["parameter_description"] = ""
for index in df.index:
    param = df.at[index, "parameter"]
    if param in parameter_list.index:
        df.at[index, "parameter_description"] = parameter_list.at[param, "description"]
df

Show the contents of the attribute `mixedSampleAnalysis`, which is a DataFrame with mixed samples (analysemonsters grond gemengd).

In [ ]:
sad.mixedSampleAnalysis

In the column `analysis` there is a DataFrame with analysis-result per sample. Let's show the results for the frist sample.

In [ ]:
sad.mixedSampleAnalysis.iloc[0]['analysis']

Show the rest of the contents of the Site Assessment Data.

In [ ]:
sad_data = sad.to_dict()
sad_data.pop("measurementPoint")
sad_data.pop("mixedSampleAnalysis")
sad_data